<a href="https://colab.research.google.com/github/jlw44874/GameWise/blob/main/GameWise.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# GameWise
My app helps budget-conscious PC gamers decide which steam game to buy by comparing games that fit their genre, budget, their playtime value, current deals, and computer compatibility.



In [22]:
from google.colab import drive
drive.mount("/content/drive")

import glob
import os
import time
import pandas as pd

# Look for steam_games.csv in My Drive and in folders up to 3 levels down
found = []
for pattern in ["", "*/", "*/*/", "*/*/*/"]:
    found = found + glob.glob("/content/drive/MyDrive/" + pattern + "steam_games.csv")

if len(found) == 0:
    print("❌ Can't find steam_games.csv in your Drive. Upload it next to this notebook and run this cell again.")
else:
    FOLDER = os.path.dirname(found[0]) + "/"
    games = pd.read_csv(FOLDER + "steam_games.csv")
    print(f"✅ Loaded {len(games)} games from:", FOLDER)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Loaded 1000 games from: /content/drive/MyDrive/MIST4600/


Sub question 1: Which games are good and match the gamer’s preferred genre and budget?



In [31]:
def find_matching_games(df, genre, budget, min_rating=80):
    in_genre = df["genres"].str.contains(genre, case=False, na=False)
    in_budget = df["price"] <= budget
    is_good = df["pct_pos_total"] >= min_rating

    matches = df[in_genre & in_budget & is_good]
    columns = ["name", "genres", "price", "pct_pos_total", "num_reviews_total"]
    return matches[columns].sort_values("pct_pos_total", ascending=False)

Code example for Sub question 1

In [24]:
matches = find_matching_games(games, genre="RPG", budget=20)
print(len(matches), "matches")
matches.head(10)

117 matches


,name,genres,price,pct_pos_total,num_reviews_total
13,Stardew Valley,"Indie, RPG, Simulation",14.99,98,733626
63,Vampire Survivors,"Action, Casual, Indie, RPG",4.99,98,234756
195,MiSide,"Adventure, Indie, RPG, Simulation",14.99,98,90341
404,VPet-Simulator,"Casual, Indie, RPG, Simulation, Free To Play",0.00,98,45250
712,Touhou Mystia's Izakaya,"Casual, Indie, RPG, Simulation",5.99,98,23658
264,OMORI,"Adventure, Indie, RPG",19.99,97,70413
289,Persona 4 Golden,RPG,19.99,97,64224
469,Volcano Princess,"Indie, RPG, Simulation",10.99,97,37456
350,South Park™: The Stick of Truth™,"Action, Adventure, RPG",7.49,97,54919
125,Mount & Blade: Warband,"Action, RPG",19.99,97,130020


Sub question 2:  Which matching games offer the most average playtime per dollar?

In [25]:
def rank_by_value(df, matches):
    games = df.loc[matches.index].copy()
    games = games[(games["price"] > 0) & (games["avg_playtime_hours"] > 0)]
    games["hours_per_dollar"] = (games["avg_playtime_hours"] / games["price"]).round(2)

    columns = ["name", "price", "avg_playtime_hours", "hours_per_dollar", "pct_pos_total"]
    return games[columns].sort_values("hours_per_dollar", ascending=False)

Code example for Sub question 2

In [26]:
best_value = rank_by_value(games, matches)
print(len(best_value), "ranked")
best_value.head(10)

59 ranked


,name,price,avg_playtime_hours,hours_per_dollar,pct_pos_total
254,FINAL FANTASY XIV Online,19.99,837.6,41.90,87
26,ARK: Survival Evolved,14.99,217.8,14.53,84
199,Brotato,2.99,39.8,13.31,96
29,PAYDAY 2,9.99,115.6,11.57,90
63,Vampire Survivors,4.99,38.2,7.66,98
56,Fallout 4,19.99,124.5,6.23,83
128,The Elder Scrolls® Online,19.99,123.4,6.17,83
140,Kingdom Come: Deliverance,7.49,44.7,5.97,83
105,Assassin's Creed® Odyssey,11.99,67.7,5.65,89
125,Mount & Blade: Warband,19.99,111.0,5.55,97


Sub question 3: Which matching games still have enough players online to be worth buying?

In [27]:
def find_active_games(df, matches, min_players=100):
    # Get the player counts for each matching game from the full data.
    games = df.loc[matches.index]

    # Active: at least min_players were online at the same time.
    is_active = games["peak_ccu"] >= min_players
    active = games[is_active]

    columns = ["name", "price", "peak_ccu", "pct_pos_total"]
    return active[columns].sort_values("peak_ccu", ascending=False)

Code example for Sub question 2

In [28]:
matches = find_matching_games(games, genre="RPG", budget=20)

active = find_active_games(games, matches)
print(len(active), "games with 100+ players online")
active.head(10)

95 games with 100+ players online


,name,price,peak_ccu,pct_pos_total
13,Stardew Valley,14.99,67309,98
22,Warframe,0.00,51688,87
39,Don't Starve Together,14.99,35283,95
31,Valheim,19.99,33903,94
64,Path of Exile,0.00,32104,89
26,ARK: Survival Evolved,14.99,31240,84
6,Terraria,9.99,30516,97
51,Project Zomboid,19.99,24694,94
239,Crusader Kings III,14.99,21948,91
56,Fallout 4,19.99,17117,83


### Outside tool: Gemini
Gemini turns a game's store description into one plain sentence
saying what the game is and who would enjoy it.

In [33]:
from google import genai
from google.colab import userdata

def describe_game(df, name):
    """Outside tool: Gemini sums up one game in a single sentence."""
    client = genai.Client(api_key=userdata.get("GEMINI_API_KEY"))

    game = df[df["name"] == name].iloc[0]
    prompt = (
        "In one short sentence, tell a gamer on a budget what this game is "
        "and who would enjoy it.\n\n"
        f"Game: {game['name']}\n"
        f"Description: {game['short_description']}"
    )
    response = client.models.generate_content(
        model="gemini-3.8-flash",
        contents=prompt,
    )
    return response.text.strip()

In [34]:
print(describe_game(games, "Stardew Valley"))

*Stardew Valley* is an affordable, cozy farming and life simulator packed with hundreds of hours of content, perfect for anyone who loves relaxing, open-ended games.
